# Hong Kong Property Data · Jupyter Notebook

Use the **hk-property-data** Python package to collect official sources, inspect the data and open company-ready Excel files. Python 3.10+ and an internet connection are required; Git and ChatGPT are not required.

Open this file in **Jupyter Notebook / JupyterLab**, select a Python kernel, and run the cells from top to bottom with **Shift + Enter**. The first download can take several minutes. Saved outputs below are a dated example; run the update cell to check for new releases.

[Dashboard](https://hogan-cyber.github.io/Real-Estate-Data/) · [Package source and guide](https://github.com/Hogan-cyber/Real-Estate-Data/blob/main/PYTHON_PACKAGE.md)

## 1. Install in this notebook

Run this cell once, or when upgrading the package. `%pip` installs into the active notebook kernel. This GitHub source archive does **not** require Git. If you already imported an older version, restart the kernel after installation, then continue from step 2.

In [1]:
%pip install --quiet --upgrade "hk-property-data @ https://github.com/Hogan-cyber/Real-Estate-Data/archive/refs/heads/main.zip" "pandas>=2.2,<4"

## 2. Choose a local output folder

Files are saved in `company-data` beside this notebook. Keep the same folder on future runs to retain revisions, history and source caches. `UPDATE_DATA = True` fetches official sources; set it to `False` to inspect an existing local snapshot without downloading.

This notebook writes local files. It does not publish changes to the shared dashboard or start a background schedule.

In [2]:
from pathlib import Path
import sqlite3
from contextlib import closing

import pandas as pd
from IPython.display import display, FileLink
from hk_property_data import refresh, load_dataset, validate, __version__

OUTPUT = Path("company-data")
UPDATE_DATA = True

pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 8)
print(f"Package version: {__version__}")
print(f"Output folder: {OUTPUT}")

Package version: 0.1.4
Output folder: company-data


## 3. Update official data

The package downloads public RVD, Land Registry, HKMA, HSBC and HKAB sources and parses them with Python. It does not use GPT. Excel, SQLite, CSV and JSON are generated together.

A source can publish later than another. A successful request does not mean every series has a new month. Status **2** means files were generated with source/comparison issues to review; it is not an installation error. A new folder has no prior revision history, so older PDF values may need comparison with revised CSV values. Fatal failures raise an exception and stop this notebook.

In [3]:
if UPDATE_DATA:
    print("Checking official sources. This can take several minutes...", flush=True)
    refresh_code = refresh(output=OUTPUT)
    if refresh_code == 0:
        print("Update complete. No unresolved source errors.")
    elif refresh_code == 2:
        print("Files generated. Review the source issues in step 4 before using the data.")
    else:
        raise RuntimeError(f"Unexpected refresh status: {refresh_code}")
elif not (OUTPUT / "hk_housing_dataset.json").is_file():
    raise FileNotFoundError("No saved dataset. Set UPDATE_DATA = True and run this cell again.")
else:
    print("Using the saved local snapshot; no sources were checked in this run.")

Checking official sources. This can take several minutes...


{"rows": 46182, "pdf_cells": 1896, "revisions": 0, "errors": ["pdf: CSV comparison needs review for tables 1.1, 1.2, 1.5, 1.3, 1.4"], "latest": "2026-10"}
Files generated. Review the source issues in step 4 before using the data.


## 4. Validate and review source checks

Validation reconciles the JSON snapshot with Excel and SQLite. **File validation and source freshness are separate checks.** Read the source issues and the dated source table before analysis.

In [4]:
validation = validate(OUTPUT)
snapshot = load_dataset(OUTPUT / "hk_housing_dataset.json")

print("File validation:", validation["validation"])
print("Snapshot checked at:", snapshot["checked_at"])
print("Source issues:")
for issue in snapshot.get("errors", []):
    print(" -", issue)
if not snapshot.get("errors"):
    print(" None")
for note in snapshot.get("warnings", []):
    print("Note:", note)

sources = pd.DataFrame(snapshot["source_status"])
source_columns = ["source_name", "status", "latest_period", "last_success"]
display(sources.reindex(columns=source_columns))

File validation: passed
Snapshot checked at: 2026-11-04T02:33:31.650322+00:00
Source issues:
 - pdf: CSV comparison needs review for tables 1.1, 1.2, 1.5, 1.3, 1.4


,source_name,status,latest_period,last_success
0,Rating and Valuation Department,Verified,2026-09,2026-11-04T02:33:31.650322+00:00
1,Rating and Valuation Department,Verified,2026-09,2026-11-04T02:33:31.650322+00:00
2,Rating and Valuation Department,Verified,2026-09,2026-11-04T02:33:31.650322+00:00
3,Rating and Valuation Department,Verified,2026-09,2026-11-04T02:33:31.650322+00:00
4,Rating and Valuation Department,Verified,2026-09,2026-11-04T02:33:31.650322+00:00
...,...,...,...,...
18,Hong Kong Monetary Authority,Verified,2026-09,2026-11-04T02:33:31.650322+00:00
19,Hong Kong Monetary Authority,Verified,2026-09,2026-11-04T02:33:31.650322+00:00
20,Rating and Valuation Department — monthly PDF,Review PDF / CSV differences,2026-08,2026-11-04T02:33:31.650322+00:00
21,HSBC effective Prime rate,Verified,2025-10-31,2026-11-04T02:33:31.650322+00:00


## 5. Work with pandas tables

`monthly` contains original monthly observations. `rates` contains separate daily HIBOR fixings and HSBC Prime effective-date changes. Dates and units remain as published; missing values remain missing.

These are **data previews**, not a comparison of different units. Do not sum indices, money supply and interest rates.

In [5]:
monthly = pd.DataFrame(snapshot["observations"])
rates = pd.DataFrame(snapshot.get("rate_observations", []))
print(f"Monthly observations: {len(monthly):,}; daily/effective rate records: {len(rates):,}")

macro_ids = ["hk_m3_hkd", "hk_prime_rate_hsbc", "hk_hibor_1m", "hk_hibor_3m"]
macro = monthly.loc[monthly["metric_id"].isin(macro_ids)].copy()
latest_monthly = macro.sort_values(["metric_id", "period"]).groupby("metric_id", sort=False).tail(1)
display(latest_monthly[["period", "metric_name", "value", "unit"]].reset_index(drop=True))

Monthly observations: 46,182; daily/effective rate records: 28


,period,metric_name,value,unit
0,2026-09,1-month HIBOR (monthly average),2.904789e+00,% p.a.
1,2026-09,3-month HIBOR (monthly average),3.105780e+00,% p.a.
2,2026-08,Hong Kong dollar M3,9.586157e+06,HKD million
3,2026-09,HSBC prime rate (monthly average),5.000000e+00,% p.a.


### Latest daily / effective rates

For HIBOR, `date` is the fixing date. For Prime, it is the effective date of the rate change; it can be old while the rate remains current. `checked_at` is the retrieval/check time, not an official publication date. Daily rates do not create extra official monthly averages. Historical 3M HIBOR coverage is limited to collected snapshots.

In [6]:
if rates.empty:
    print("No daily/effective rates are available. Review source checks above.")
else:
    latest_rates = rates.sort_values(["metric_id", "date"]).groupby("metric_id", sort=False).tail(1)
    display(latest_rates[["date", "metric_name", "value", "unit", "observation_basis", "checked_at"]].reset_index(drop=True))

,date,metric_name,value,unit,observation_basis,checked_at
0,2026-11-03,1 Month HIBOR,2.68000,% p.a.,Daily fixing,2026-11-04T02:33:31.650322+00:00
1,2026-11-03,3 Months HIBOR,3.21399,% p.a.,Daily fixing,2026-11-04T02:33:31.650322+00:00
2,2025-10-31,HSBC Prime rate,5.00000,% p.a.,Effective rate,2026-11-04T02:33:31.650322+00:00


## 6. Query residential sale prices with SQL

Edit the district and month range below. A–E are **saleable-area classes**, not bedroom counts. `price_hkd_per_sqft` converts the official HKD/sq m averages by 0.09290304; the source excludes primary residential sales. Missing observations and source flags are preserved. The query uses read-only SQLite access.

In [7]:
REGION = "Kowloon"   # "Hong Kong Island", "Kowloon", or "New Territories"
START_MONTH = "2025-01"
END_MONTH = "2026-12"

sql = """
SELECT period, class_name, price_hkd_per_sqft, flags
FROM residential_sale_prices
WHERE region = ? AND period BETWEEN ? AND ?
ORDER BY period DESC, class_name
"""
database_uri = (OUTPUT / "hk_housing.sqlite").resolve().as_uri() + "?mode=ro"
with closing(sqlite3.connect(database_uri, uri=True)) as connection:
    prices = pd.read_sql_query(sql, connection, params=(REGION, START_MONTH, END_MONTH))

print(f"Matching rows: {len(prices):,}")
display(prices.head(10))

Matching rows: 105


,period,class_name,price_hkd_per_sqft,flags
0,2026-09,A,12480.315684,P
1,2026-09,B,12683.959148,P
2,2026-09,C,16070.553665,P
3,2026-09,D,17512.223040,P
4,2026-09,E,24473.726536,PZ
5,2026-08,A,12053.426216,P
6,2026-08,B,12672.717880,P
7,2026-08,C,16787.300619,P
8,2026-08,D,19876.233796,PZ
9,2026-08,E,23888.344481,PZ


## 7. Download Excel for colleagues

The full workbook contains **all available periods**, not just the SQL selection. `selected_residential_prices.xlsx` below contains only the query result from step 6. Both are local files generated by this notebook. Click the links, or find the files in Jupyter's `company-data` folder.

In [8]:
selected_file = OUTPUT / "selected_residential_prices.xlsx"
prices.to_excel(selected_file, index=False, engine="xlsxwriter")

for filename in ["hk_property_data.xlsx", "selected_residential_prices.xlsx", "hk_housing.sqlite", "refresh_status.json"]:
    path = OUTPUT / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    display(FileLink(str(path)))

company-data/hk_property_data.xlsx

company-data/selected_residential_prices.xlsx

company-data/hk_housing.sqlite

company-data/refresh_status.json

## Future updates

- To fetch new releases, keep the same `company-data` folder, set `UPDATE_DATA = True`, and rerun steps 2–7. Installation is only needed for a new environment or a code upgrade.
- To analyse saved data offline, set `UPDATE_DATA = False` and rerun steps 2–7. Do not run the online installation cell while offline.
- Closing Jupyter stops the notebook; it does not create a background update schedule. The shared dashboard's existing GitHub Actions job continues independently.
- Reusing the same folder preserves local revisions. Do not run two refreshes against it concurrently.
- Company deployments can pin a reviewed commit or use the bundled `hk_property_data-0.1.4-py3-none-any.whl` with `%pip install "hk_property_data-0.1.4-py3-none-any.whl" "pandas>=2.2,<4"` in place of step 1. Dependency downloads still require internet unless already installed.

Sources and definitions remain in the saved dataset, the Excel **Source Checks** sheet, and the [database guide](https://github.com/Hogan-cyber/Real-Estate-Data/blob/main/DATABASE.md).